# Baseline: NeCSTGen

NeCSTGen (Meslet-Millet et al., GLOBECOM 2022): a variational autoencoder compresses single packets into a latent space, a Bayesian Gaussian mixture models the latent codes, and a bidirectional GRU predicts the IAT of a packet from the previous 11 packets. For every test flow, one latent code per packet is drawn from the mixture and decoded; the payload is taken from the decoder and, after the first 11 packets, the IAT from the recurrent predictor. All computations use float64, as in the original implementation.

Run by `scripts/run_matrix.py` after the proposed model of the same dataset and seed, whose `split.json` and `test.csv` it reuses. The packets are cleaned with the same function as for the proposed model, and the run stops if its test flows differ from `test.csv`.

In [ ]:
SEED         = 0
DATASET      = 'facebook_video'
RAW_PATH_STR = 'data/facebook_video.csv'
OUT_DIR      = 'runs/facebook_video/seed0'
MODEL_KEY    = 'necstgen'
TEST_SIZE    = 0.3              # not used: the split is read from split.json

In [ ]:
import os
import random
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.mixture import BayesianGaussianMixture
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()   # run from the repository root
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'scripts'))
from data_utils import atomic_write_csv, load_packets, load_split_checked, postprocess_synthetic

ART = Path(OUT_DIR) / 'artifacts' / MODEL_KEY
ART.mkdir(parents=True, exist_ok=True)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':16:8'
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.set_default_dtype(torch.float64)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

TIMESTEPS   = 11      # packets of context of the recurrent predictor
LATENT_DIM  = 4       # latent dimension of the VAE
EPOCHS_VAE  = 50
EPOCHS_LSTM = 50
BATCH_SIZE  = 64


def create_windows(data, window_shape, step=1, start_id=None, end_id=None):
    # sliding windows (N, window_shape, features) by stride tricks, as in the NeCSTGen code
    data = np.asarray(data)
    data = data.reshape(-1, 1) if np.prod(data.shape) == max(data.shape) else data
    start_id = 0 if start_id is None else start_id
    end_id = data.shape[0] if end_id is None else end_id
    data = data[int(start_id):int(end_id), :]
    window_shape = (int(window_shape), data.shape[-1])
    step = (int(step),) * data.ndim
    slices = tuple(slice(None, None, st) for st in step)
    indexing_strides = data[slices].strides
    win_indices_shape = ((np.array(data.shape) - window_shape) // step) + 1
    new_shape = tuple(list(win_indices_shape) + list(window_shape))
    strides = tuple(list(indexing_strides) + list(data.strides))
    return np.squeeze(np.lib.stride_tricks.as_strided(data, shape=new_shape, strides=strides), 1)


def standardize(x, min_x, max_x, a=0, b=1):
    # min-max scaling of x into [a, b]
    if max_x == min_x:
        return np.zeros_like(x)
    return (b - a) * ((x - min_x) / (max_x - min_x)) + a

In [ ]:
df = load_packets(RAW_PATH_STR)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    df['payload_log'] = np.log1p(df['payload_length'].values)
    df['time_log']    = np.log1p(df['time_diff'].values)
train_flows, test_flows = load_split_checked(df, OUT_DIR)
train_df = df[df['flow_id'].isin(train_flows)].copy()
min_p, max_p = train_df['payload_log'].min(), train_df['payload_log'].max()
min_t, max_t = train_df['time_log'].min(), train_df['time_log'].max()
train_df['p_scaled'] = standardize(train_df['payload_log'], min_p, max_p)
train_df['t_scaled'] = standardize(train_df['time_log'], min_t, max_t)

# windows of TIMESTEPS packets and the packet that follows them, for the recurrent predictor
X_seq_list, Y_seq_list = [], []
for fid, group in tqdm(train_df.groupby('flow_id'), desc='windows'):
    vals = group[['p_scaled', 't_scaled']].values
    if len(vals) <= TIMESTEPS:
        continue
    windows = create_windows(vals, window_shape=TIMESTEPS + 1)
    X_seq_list.append(windows[:, :-1, :])           # (N, 11, 2)
    Y_seq_list.append(windows[:, -1, :])            # (N, 2)
X_train_seq = torch.tensor(np.vstack(X_seq_list), dtype=torch.float64)
Y_train_seq = torch.tensor(np.vstack(Y_seq_list), dtype=torch.float64)
X_train_flat = torch.tensor(train_df[['p_scaled', 't_scaled']].values, dtype=torch.float64)   # single packets for the VAE
print(f'windows {tuple(X_train_seq.shape)}, packets {tuple(X_train_flat.shape)}')

In [ ]:
class NeCSTGenEncoder(nn.Module):
    def __init__(self, input_dim=2, latent_dim=4):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(input_dim, 8), nn.LeakyReLU(0.1), nn.Linear(8, 6), nn.LeakyReLU(0.1),
                                    nn.Linear(6, 4), nn.LeakyReLU(0.1))
        self.z_mean = nn.Linear(4, latent_dim)
        self.z_log_var = nn.Linear(4, latent_dim)

    def forward(self, x):
        h = self.shared(x)
        return self.z_mean(h), self.z_log_var(h)


class NeCSTGenDecoder(nn.Module):
    def __init__(self, latent_dim=4, output_dim=2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(latent_dim, 4), nn.LeakyReLU(0.1), nn.Linear(4, 6), nn.LeakyReLU(0.1),
                                 nn.Linear(6, 8), nn.LeakyReLU(0.1), nn.Linear(8, output_dim), nn.Sigmoid())

    def forward(self, z):
        return self.net(z)


class NeCSTGenVAE(nn.Module):
    def __init__(self, input_dim=2, latent_dim=4):
        super().__init__()
        self.encoder = NeCSTGenEncoder(input_dim, latent_dim)
        self.decoder = NeCSTGenDecoder(latent_dim, input_dim)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        return self.decoder(self.reparameterize(mu, logvar)), mu, logvar


def necstgen_vae_loss(recon_x, x, mu, logvar, beta=0.01):
    # reconstruction (binary cross-entropy plus L1) and a KL term weighted by beta; with
    # beta = 1 the KL term collapses the two-dimensional data onto its mean
    bce = F.binary_cross_entropy(recon_x, x, reduction='none').sum(dim=1)
    mae = F.l1_loss(recon_x, x, reduction='none').sum(dim=1)
    recon_loss = bce + mae
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
    return torch.mean(recon_loss + beta * kl_loss)


vae = NeCSTGenVAE(input_dim=2, latent_dim=LATENT_DIM).to(device)
optimizer = optim.Adam(vae.parameters(), lr=1e-3)
dataloader_flat = DataLoader(TensorDataset(X_train_flat), batch_size=BATCH_SIZE, shuffle=True)
vae.train()
for epoch in range(EPOCHS_VAE):
    total_loss = 0
    for batch in dataloader_flat:
        x = batch[0].to(device)
        optimizer.zero_grad()
        recon_x, mu, logvar = vae(x)
        loss = necstgen_vae_loss(recon_x, x, mu, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f'VAE epoch {epoch + 1}/{EPOCHS_VAE}: loss {total_loss / len(dataloader_flat):.4f}')
torch.save(vae.state_dict(), ART / 'vae_weights.pth')

# Bayesian Gaussian mixture of the latent codes (settings of the NeCSTGen code)
vae.eval()
with torch.no_grad():
    mu, logvar = vae.encoder(X_train_flat.to(device))
    latents = vae.reparameterize(mu, logvar).cpu().numpy()
bgm = BayesianGaussianMixture(n_components=70, weight_concentration_prior=1000, covariance_type='full',
                              init_params='kmeans', max_iter=1000, n_init=1, random_state=SEED)
bgm.fit(latents)
# with an integer random_state, sample() restarts its generator on every call, so that all flows of
# the same length would be identical; a RandomState instance advances from call to call
bgm.random_state = np.random.RandomState(SEED)
joblib.dump(bgm, ART / 'bgm_model.sav')

In [ ]:
class NeCSTGenPredictor(nn.Module):
    # bidirectional GRU over the last TIMESTEPS packets; the output of the last step gives the next packet
    def __init__(self, input_dim=2, hidden_dim=4, output_dim=2):
        super().__init__()
        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        gru_out, _ = self.gru(x)
        return self.sigmoid(self.fc(gru_out[:, -1, :]))


predictor = NeCSTGenPredictor(input_dim=2, hidden_dim=LATENT_DIM, output_dim=2).to(device)
criterion = nn.MSELoss()
optimizer_pred = optim.Adam(predictor.parameters(), lr=1e-3)
dataloader_seq = DataLoader(TensorDataset(X_train_seq, Y_train_seq), batch_size=BATCH_SIZE, shuffle=True)
predictor.train()
for epoch in range(EPOCHS_LSTM):
    total_loss = 0
    for batch_x, batch_y in dataloader_seq:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer_pred.zero_grad()
        loss = criterion(predictor(batch_x), batch_y)
        loss.backward()
        optimizer_pred.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f'predictor epoch {epoch + 1}/{EPOCHS_LSTM}: MSE {total_loss / len(dataloader_seq):.6f}')
torch.save(predictor.state_dict(), ART / 'predictor_weights.pth')

In [ ]:
vae.eval()
predictor.eval()
records = []
perm_rng = np.random.default_rng(SEED + 7)
for fid, group in tqdm(df[df['flow_id'].isin(test_flows)].groupby('flow_id'), desc='flows'):
    L = len(group)
    if L == 0:
        continue
    with torch.no_grad():
        latents_np, _ = bgm.sample(L)                         # one latent code per packet
        latents_np = latents_np[perm_rng.permutation(L)]      # sample() returns the draws grouped by component
        vae_features = vae.decoder(torch.tensor(latents_np, dtype=torch.float64, device=device))   # (L, 2)
        final_p_scaled = vae_features[:, 0].cpu().numpy()     # payload from the decoder
        final_t_scaled = vae_features[:, 1].cpu().numpy()
        if L > TIMESTEPS:                                     # IAT of the later packets from the predictor
            ctx = vae_features[:TIMESTEPS].unsqueeze(0)
            for i in range(TIMESTEPS, L):
                pred = predictor(ctx)
                final_t_scaled[i] = pred[0, 1].cpu().numpy()
                ctx = torch.cat([ctx[:, 1:, :], vae_features[i].view(1, 1, 2)], dim=1)
    for i in range(L):
        payload = np.expm1(final_p_scaled[i] * (max_p - min_p) + min_p)
        time_dt = np.expm1(final_t_scaled[i] * (max_t - min_t) + min_t)
        records.append([fid, round(float(np.clip(payload, 0.0, 1500.0))), float(max(time_dt, 0.0))])
synth_df = pd.DataFrame(records, columns=['flow_id', 'payload_length', 'time_diff'])

joblib.dump({'min_p': min_p, 'max_p': max_p, 'min_t': min_t, 'max_t': max_t}, ART / 'necstgen_bounds.pkl')

synth_df = postprocess_synthetic(synth_df)
atomic_write_csv(synth_df, Path(OUT_DIR) / f'{MODEL_KEY}.csv')
print(f'saved {len(synth_df):,} packets in {synth_df.flow_id.nunique()} flows')